In [1]:
import glob
import numpy as np
import time
from PIL import Image
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import MulticlassConfusionMatrix
from torch.utils.data import Dataset, DataLoader

In [2]:
CLASSES = [ 
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', 'Misc'
]

colors = ['#ffffff','#3a3b7b','#6a6ecf','#8ca351','#fff100','#ff00ff','#833c39','#e598a0']

RESIZE_TO = 640 

ANCHORS = [
    # LARGE OBJECT SCALE (S=20)
    [
        (0.0737, 0.1407),
        (0.1173, 0.2590),
        (0.2270, 0.4612),
    ],
    # SMALL OBJECT SCALE (S=40)
    [
        (0.0223, 0.0665),
        (0.0420, 0.0937),
        (0.0294, 0.2288),
    ],
]

S = [RESIZE_TO // 32, RESIZE_TO // 16]

NUM_CLASSES = len(CLASSES) 
NUM_WORKERS = 4
BATCH_SIZE = 5 * 2
RESIZE_TO = 320 
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4 
PIN_MEMORY = True
AMP = True
EPOCHS = 100

CHECKPOINT_FILE_42 = "./checkpoints/best_full_42.pth.tar"
CHECKPOINT_FILE_123 = "./checkpoints/best_full_123.pth.tar"
CHECKPOINT_FILE_999 = "./checkpoints/best_full_999.pth.tar"

BASE_DATASET_PATH = '../../datasets/KITTI/dataset'
test_images_path = f'{BASE_DATASET_PATH}/test/images/'

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
""" 
Information about architecture config:
"B" indicating a residual block
"S" is for scale prediction block
"U" is for upsampling the feature map
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 4],
    (128, 3, 2),
    ["B", 6],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 6],
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
]

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers += [
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            ]

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)

        return x


class ScalePrediction(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels, 3 * (num_classes + 5), bn_act=False, kernel_size=1
            ),
        )
        
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

class SiStNet(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []
        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels
        
        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels=in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3

        return layers

if __name__ == "__main__": 
    model = SiStNet(num_classes=NUM_CLASSES)

In [5]:

test_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=RESIZE_TO),
        A.PadIfNeeded(
            min_height=RESIZE_TO, min_width=RESIZE_TO, border_mode=cv2.BORDER_CONSTANT
        ),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255,),
        ToTensorV2(),
    ],
)

In [6]:
model = SiStNet(num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler(
    enabled=AMP,
    init_scale=2**12
)

# ---------------------------
# 3. CHECKPOINT LOAD
# ---------------------------
def load_model(checkpoint_path, model, optimizer, scheduler, scaler):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)    
    
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    if scheduler and checkpoint["scheduler"]:
        scheduler.load_state_dict(checkpoint["scheduler"])

    if scaler and checkpoint["scaler"]:
        scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    best_map = checkpoint["best_map"]

    seed = checkpoint["seed"]

    #torch_rng_state = torch.set_rng_state(checkpoint["torch_rng_state"])
    #cuda_rng_state = torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
    #numpy_rng_state = np.random.set_state(checkpoint["numpy_rng_state"])
    #python_rng_state = random.setstate(checkpoint["python_rng_state"])

    print("Full Checkpoint loaded!")
    
    return start_epoch, best_map, seed
    

# ---------------------------
# 4. INFERENCE DATASET (LABELSIZ)
# ---------------------------
class InferenceDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.images = glob.glob(f"{img_dir}/*")
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, img_path


In [7]:
import time
import numpy as np
import torch

# ---------------------------
# CUDA OPTIMIZATION
# ---------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False


# ---------------------------
# PER-IMAGE BENCHMARK (METHOD 1)
# ---------------------------
def run_per_image_benchmark(model, loader, device):
    model.eval()

    latencies = []
    total_images = 0

    # ---------------- WARMUP ----------------
    with torch.no_grad():
        for i in range(10):
            img = next(iter(loader))[0][0].unsqueeze(0).to(device)
            _ = model(img)

    if device == "cuda":
        torch.cuda.synchronize()

    # ---------------- TIMED ----------------
    with torch.no_grad():
        for images, _ in loader:

            for img in images:
                img = img.unsqueeze(0).to(device)

                if device == "cuda":
                    torch.cuda.synchronize()

                start = time.perf_counter()
                _ = model(img)

                if device == "cuda":
                    torch.cuda.synchronize()

                end = time.perf_counter()

                latency = (end - start) * 1000
                latencies.append(latency)
                total_images += 1

                print(f"Image | {latency:.2f} ms | {1000/latency:.2f} FPS")

    total_time = sum(latencies) / 1000

    print("\n====================")
    print("FINAL RESULTS")
    print("====================")
    print(f"Avg Latency: {np.mean(latencies):.2f} ms")
    print(f"Throughput FPS: {total_images / total_time:.2f}")

    return total_images / total_time, np.mean(latencies)

In [8]:
# ---------------------------
# 6. DATASET + DATALOADER
# ---------------------------
test_dataset = InferenceDataset(
    img_dir=test_images_path,
    transform=test_transforms
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ---------------------------
# 7. ANCHORS
# ---------------------------
scaled_anchors = [
    torch.tensor(ANCHORS[i], device=DEVICE, dtype=torch.float32) * float(S[i])
    for i in range(len(S))
]



In [9]:
def run_batch_benchmark(model, loader, device):
    model.eval()

    total_time = 0.0
    total_images = 0

    # ---------------- WARMUP ----------------
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            if i == 5:
                break

    if device == "cuda":
        torch.cuda.synchronize()

    # ---------------- TIMED ----------------
    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(loader):

            images = images.to(device)
            batch_size = images.shape[0]

            if device == "cuda":
                torch.cuda.synchronize()

            start = time.perf_counter()
            _ = model(images)
            if device == "cuda":
                torch.cuda.synchronize()
            end = time.perf_counter()

            batch_time = end - start

            total_time += batch_time
            total_images += batch_size

            batch_fps = batch_size / batch_time
            latency = (batch_time / batch_size) * 1000

            print(
                f"Batch {batch_idx} | "
                f"Batch FPS: {batch_fps:.2f} | "
                f"Latency: {latency:.2f} ms/img"
            )

    avg_fps = total_images / total_time
    avg_latency = (total_time / total_images) * 1000

    print("\n====================")
    print("FINAL BATCH RESULTS")
    print("====================")
    print(f"Avg FPS (throughput): {avg_fps:.2f}")
    print(f"Avg Latency: {avg_latency:.2f} ms/image")

    return avg_fps, avg_latency

In [10]:

start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_42, model, optimizer, scheduler, scaler)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)


Full Checkpoint loaded!
Image | 26.15 ms | 38.25 FPS
Image | 25.74 ms | 38.85 FPS
Image | 25.71 ms | 38.89 FPS
Image | 25.63 ms | 39.02 FPS
Image | 25.53 ms | 39.18 FPS
Image | 25.55 ms | 39.14 FPS
Image | 25.52 ms | 39.18 FPS
Image | 25.44 ms | 39.32 FPS
Image | 25.39 ms | 39.38 FPS
Image | 25.50 ms | 39.21 FPS
Image | 25.19 ms | 39.70 FPS
Image | 19.52 ms | 51.22 FPS
Image | 19.05 ms | 52.50 FPS
Image | 18.97 ms | 52.71 FPS
Image | 18.94 ms | 52.78 FPS
Image | 18.97 ms | 52.71 FPS
Image | 18.93 ms | 52.82 FPS
Image | 18.96 ms | 52.73 FPS
Image | 18.61 ms | 53.73 FPS
Image | 19.08 ms | 52.42 FPS

FINAL RESULTS
Avg Latency: 22.62 ms
Throughput FPS: 44.21
Batch 0 | Batch FPS: 83.02 | Latency: 12.05 ms/img
Batch 1 | Batch FPS: 80.82 | Latency: 12.37 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 81.91
Avg Latency: 12.21 ms/image


In [11]:

start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_123, model, optimizer, scheduler, scaler)

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

Full Checkpoint loaded!
Batch 0 | Batch FPS: 83.10 | Latency: 12.03 ms/img
Batch 1 | Batch FPS: 81.35 | Latency: 12.29 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 82.22
Avg Latency: 12.16 ms/image
Image | 26.35 ms | 37.96 FPS
Image | 25.82 ms | 38.73 FPS
Image | 25.67 ms | 38.96 FPS
Image | 25.63 ms | 39.01 FPS
Image | 25.60 ms | 39.06 FPS
Image | 25.62 ms | 39.04 FPS
Image | 25.32 ms | 39.50 FPS
Image | 25.29 ms | 39.54 FPS
Image | 25.31 ms | 39.50 FPS
Image | 24.40 ms | 40.98 FPS
Image | 18.67 ms | 53.55 FPS
Image | 18.46 ms | 54.17 FPS
Image | 18.70 ms | 53.49 FPS
Image | 18.85 ms | 53.04 FPS
Image | 18.87 ms | 52.99 FPS
Image | 18.87 ms | 52.99 FPS
Image | 18.88 ms | 52.97 FPS
Image | 18.96 ms | 52.75 FPS
Image | 18.67 ms | 53.57 FPS
Image | 19.14 ms | 52.24 FPS

FINAL RESULTS
Avg Latency: 22.15 ms
Throughput FPS: 45.14


In [12]:
start_epoch, best_map, seed = load_model(CHECKPOINT_FILE_999, model, optimizer, scheduler, scaler)

avg_fps, avg_latency = run_batch_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

# ---------------------------
# 8. RUN BENCHMARK
# ---------------------------
avg_fps, avg_latency = run_per_image_benchmark(
    model=model,
    loader=test_loader,
    device=DEVICE
)

Full Checkpoint loaded!
Batch 0 | Batch FPS: 83.09 | Latency: 12.04 ms/img
Batch 1 | Batch FPS: 81.06 | Latency: 12.34 ms/img

FINAL BATCH RESULTS
Avg FPS (throughput): 82.06
Avg Latency: 12.19 ms/image
Image | 26.04 ms | 38.40 FPS
Image | 25.72 ms | 38.87 FPS
Image | 25.69 ms | 38.92 FPS
Image | 25.52 ms | 39.18 FPS
Image | 25.49 ms | 39.23 FPS
Image | 25.50 ms | 39.22 FPS
Image | 22.52 ms | 44.40 FPS
Image | 18.93 ms | 52.83 FPS
Image | 18.71 ms | 53.44 FPS
Image | 18.80 ms | 53.20 FPS
Image | 19.00 ms | 52.63 FPS
Image | 18.91 ms | 52.88 FPS
Image | 18.91 ms | 52.87 FPS
Image | 18.93 ms | 52.84 FPS
Image | 18.88 ms | 52.97 FPS
Image | 18.64 ms | 53.65 FPS
Image | 19.04 ms | 52.52 FPS
Image | 19.04 ms | 52.52 FPS
Image | 19.09 ms | 52.39 FPS
Image | 19.05 ms | 52.48 FPS

FINAL RESULTS
Avg Latency: 21.12 ms
Throughput FPS: 47.35
